# Data Preprocessing & Export Pipeline Protocol
## UCI Machine Learning Drug Review Dataset (KUC Hackathon Winter 2018)



#### Pipeline Sequential Stages:
1. **Data Ingestion:** Load raw train (`drugsComTrain_raw.tsv`) and test (`drugsComTest_raw.tsv`) datasets.
2. **Text Cleaning & Quote Sanitization:** Strip outer quotation marks (`"`, `'`), linebreaks, and HTML entities (`html.unescape`).
3. **Contradiction Edge-Case Extraction & Train Sanitization:**
   - Extract rows with identical review text but conflicting rating scores into `explainability_edge_cases.csv` for the GenAI Explainability phase.
   - Drop these contradictory rows from `train_df` to prevent noisy gradient signals during training.
4. **Hierarchical Condition Extraction (Option B):** Parse conditions into `condition_primary`, `condition_subtype`, and `condition_clean` without row explosion.
5. **Feature Engineering & Target Discretization:** Discretize 3-class target (`rating_3_class`), compute log usefulness, text length metrics, and date decomposition.
6. **Leakage & Validation Strategy Documentation:** Explicitly document Option B (Grouped K-Fold Cross Validation by review text).
7. **Data Verification & Disk Export:** Export `cleaned_train.csv`, `cleaned_test.csv`, and `explainability_edge_cases.csv` and verify disk sizes.

In [53]:
import pandas as pd
import numpy as np
import html
import re
import os
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

print("[INFO] Scientific libraries initialized for preprocessing pipeline.")

# Dynamic path resolution
train_path = 'data/drugsComTrain_raw.tsv' if os.path.exists('data/drugsComTrain_raw.tsv') else 'data/drugsComTrain_raw.csv'
test_path = 'data/drugsComTest_raw.tsv' if os.path.exists('data/drugsComTest_raw.tsv') else 'data/drugsComTest_raw.csv'

sep = '\t' if train_path.endswith('.tsv') else ','

print(f"[INFO] Ingesting raw training data: {train_path}")
train_df = pd.read_csv(train_path, sep=sep)

print(f"[INFO] Ingesting raw testing data:  {test_path}")
test_df = pd.read_csv(test_path, sep=sep)

print(f"\nInitial Training Matrix: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Initial Testing Matrix:  {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")

print("\nRaw Data Sample:")
train_df.head(3)

[INFO] Scientific libraries initialized for preprocessing pipeline.
[INFO] Ingesting raw training data: data/drugsComTrain_raw.csv
[INFO] Ingesting raw testing data:  data/drugsComTest_raw.csv

Initial Training Matrix: 161,297 rows x 7 columns
Initial Testing Matrix:  53,766 rows x 7 columns

Raw Data Sample:


,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17


## 2. Text Cleaning, Quote Sanitization & Hierarchical Condition Extraction Stage

In [54]:
# Text Sanitization & Contradiction Extraction Functions
def sanitize_review(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = html.unescape(text)
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    quote_chars = '"\'\u201c\u201d\u2018\u2019'
    text = text.strip(quote_chars)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def sanitize_condition(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = html.unescape(text)
    return text.strip()

def parse_condition_hierarchy(cond):
    if pd.isna(cond) or not str(cond).strip():
        return pd.Series(['Unknown', '', 'Unknown'])
    cond_str = str(cond).strip()
    if ',' in cond_str:
        parts = [p.strip() for p in cond_str.split(',', 1)]
        primary = parts[0]
        subtype = parts[1] if len(parts) > 1 else ''
        clean = f"{subtype} {primary}".strip() if subtype else primary
    else:
        primary = cond_str
        subtype = ''
        clean = cond_str
    return pd.Series([primary, subtype, clean])

print("[1/4] Stripping quotes, HTML tags, decoding entities, and removing missing values...")
train_df['condition'] = train_df['condition'].apply(sanitize_condition)
test_df['condition'] = test_df['condition'].apply(sanitize_condition)
train_df['review'] = train_df['review'].apply(sanitize_review)
test_df['review'] = test_df['review'].apply(sanitize_review)

train_df = train_df.dropna(subset=['review', 'condition', 'rating']).copy()
test_df = test_df.dropna(subset=['review', 'condition', 'rating']).copy()

scraping_pattern = 'users found this comment helpful'
train_df = train_df[~train_df['condition'].str.contains(scraping_pattern, case=False, na=False)].copy()
test_df = test_df[~test_df['condition'].str.contains(scraping_pattern, case=False, na=False)].copy()

print("[2/4] Isolating Contradictory Edge Cases for GenAI & Sanitizing Training Set...")
review_rating_nunique = train_df.groupby('review')['rating'].nunique()
contradictory_reviews = review_rating_nunique[review_rating_nunique > 1]
explainability_edge_cases = train_df[train_df['review'].isin(contradictory_reviews.index)].copy()

print(f"  - Extracted {len(explainability_edge_cases):,} contradictory rows ({len(contradictory_reviews):,} unique reviews) for GenAI Explainability.")

# Drop contradictory rows from train_df to eliminate noisy gradient updates
train_df = train_df[~train_df['review'].isin(contradictory_reviews.index)].copy()
print(f"  - Sanitized Training Set size: {len(train_df):,} rows")

print("[3/4] Extracting Hierarchical Condition Features (Option B)... ")
train_df[['condition_primary', 'condition_subtype', 'condition_clean']] = train_df['condition'].apply(parse_condition_hierarchy)
test_df[['condition_primary', 'condition_subtype', 'condition_clean']] = test_df['condition'].apply(parse_condition_hierarchy)
explainability_edge_cases[['condition_primary', 'condition_subtype', 'condition_clean']] = explainability_edge_cases['condition'].apply(parse_condition_hierarchy)

print("[INFO] Text Sanitization, Contradiction Extraction & Hierarchical Parsing Complete!")
display(train_df[['condition', 'condition_primary', 'condition_subtype', 'condition_clean', 'review']].head(3))


[1/4] Stripping quotes, HTML tags, decoding entities, and removing missing values...
[2/4] Isolating Contradictory Edge Cases for GenAI & Sanitizing Training Set...
  - Extracted 407 contradictory rows (75 unique reviews) for GenAI Explainability.
  - Sanitized Training Set size: 159,091 rows
[3/4] Extracting Hierarchical Condition Features (Option B)... 
[INFO] Text Sanitization, Contradiction Extraction & Hierarchical Parsing Complete!


,condition,condition_primary,condition_subtype,condition_clean,review
0,Left Ventricular Dysfunction,Left Ventricular Dysfunction,,Left Ventricular Dysfunction,"It has no side effect, I take it in combinatio..."
1,ADHD,ADHD,,ADHD,My son is halfway through his fourth week of I...
2,Birth Control,Birth Control,,Birth Control,"I used to take another oral contraceptive, whi..."


## 3. Feature Engineering & Target Discretization Stage

In [55]:
def map_rating_to_3_class(rating):
    if rating <= 4:
        return 1
    elif rating < 7:
        return 2
    else:
        return 3

for df_target in [train_df, test_df, explainability_edge_cases]:
    df_target['rating_3_class'] = df_target['rating'].apply(map_rating_to_3_class)
    df_target['log_usefulCount'] = np.log1p(df_target['usefulCount'])
    df_target['review_length'] = df_target['review'].apply(lambda x: len(str(x).split()))
    df_target['est_token_count'] = (df_target['review_length'] * 1.3).astype(int)
    df_target['char_length'] = df_target['review'].apply(len)
    df_target['avg_word_length'] = (df_target['char_length'] / (df_target['review_length'] + 1e-5)).round(2)
    df_target['date_dt'] = pd.to_datetime(df_target['date'], format='%d-%b-%y')
    df_target['year'] = df_target['date_dt'].dt.year
    df_target['month'] = df_target['date_dt'].dt.month
    df_target.drop(columns=['date_dt'], inplace=True)

print("=== CLEANED & FEATURE-ENGINEERED SCHEMA PREVIEW ===")
display(train_df.head(3))


=== CLEANED & FEATURE-ENGINEERED SCHEMA PREVIEW ===


,uniqueID,drugName,condition,review,rating,date,usefulCount,condition_primary,condition_subtype,condition_clean,rating_3_class,log_usefulCount,review_length,est_token_count,char_length,avg_word_length,year,month
0,206461,Valsartan,Left Ventricular Dysfunction,"It has no side effect, I take it in combinatio...",9,20-May-12,27,Left Ventricular Dysfunction,,Left Ventricular Dysfunction,3,3.332205,17,22,77,4.53,2012,5
1,95260,Guanfacine,ADHD,My son is halfway through his fourth week of I...,8,27-Apr-10,192,ADHD,,ADHD,3,5.262690,141,183,737,5.23,2010,4
2,92703,Lybrel,Birth Control,"I used to take another oral contraceptive, whi...",5,14-Dec-09,17,Birth Control,,Birth Control,2,2.890372,134,174,742,5.54,2009,12


## 4. Test Set Leakage Protocol & Grouped Cross-Validation Strategy



## 5. Final Dataset Export & Disk Artifact Verification Stage


In [56]:
cleaned_train_path = 'cleaned_train.csv'
cleaned_test_path = 'cleaned_test.csv'
edge_cases_path = 'explainability_edge_cases.csv'

train_df.to_csv(cleaned_train_path, index=False)
test_df.to_csv(cleaned_test_path, index=False)
explainability_edge_cases.to_csv(edge_cases_path, index=False)

print("=== DATASET EXPORT PIPELINE COMPLETE ===")
print(f"Exported Cleaned Training Set:         '{cleaned_train_path}' ({train_df.shape[0]:,} rows, {train_df.shape[1]} columns)")
print(f"Exported Cleaned Testing Set:          '{cleaned_test_path}' ({test_df.shape[0]:,} rows, {test_df.shape[1]} columns)")
print(f"Exported GenAI Explainability Artifact: '{edge_cases_path}' ({explainability_edge_cases.shape[0]:,} rows, {explainability_edge_cases.shape[1]} columns)")

print(f"\nVerified Disk Artifact Sizes:")
print(f"  - '{cleaned_train_path}': {os.path.getsize(cleaned_train_path) / (1024 * 1024):.2f} MB")
print(f"  - '{cleaned_test_path}':  {os.path.getsize(cleaned_test_path) / (1024 * 1024):.2f} MB")
print(f"  - '{edge_cases_path}': {os.path.getsize(edge_cases_path) / (1024 * 1024):.2f} MB")


=== DATASET EXPORT PIPELINE COMPLETE ===
Exported Cleaned Training Set:         'cleaned_train.csv' (159,091 rows, 18 columns)
Exported Cleaned Testing Set:          'cleaned_test.csv' (53,200 rows, 18 columns)
Exported GenAI Explainability Artifact: 'explainability_edge_cases.csv' (407 rows, 18 columns)

Verified Disk Artifact Sizes:
  - 'cleaned_train.csv': 86.92 MB
  - 'cleaned_test.csv':  28.98 MB
  - 'explainability_edge_cases.csv': 0.06 MB
